In [2]:
# --- 1. Define the Comparison Tool ---

In [1]:
import numpy as np
import pandas as pd

def compare_two_cells(model, group_A, group_B):
    """
    Compare the statistical difference between two specific groups based on a fitted GEE model.
    
    Parameters:
    model: The fitted statsmodels GEE result object (e.g., gee_model_interact)
    group_A: Dictionary of conditions for the first cell (e.g., {'food_1.0': 1, 'varroa_1': 1})
    group_B: Dictionary of conditions for the second cell (e.g., {'food_3.0': 1, 'varroa_5': 1})
    """
    # Get the exact names of the features used in the model
    feature_names = model.model.exog_names
    
    # Initialize empty vectors for Group A and Group B (all zeros)
    vector_A = np.zeros(len(feature_names))
    vector_B = np.zeros(len(feature_names))
    
    # Helper function to fill the vectors including interaction terms
    def fill_vector(vector, conditions):
        # 1. Always set Intercept (const) to 1
        if 'const' in feature_names:
            vector[feature_names.index('const')] = 1
            
        # 2. Set main effects
        for key, val in conditions.items():
            if key in feature_names:
                vector[feature_names.index(key)] = val
                
        # 3. Automatically activate interaction terms if both main effects are present
        for i, feature in enumerate(feature_names):
            if '_X_' in feature: # Assuming your interaction terms have '_X_'
                parts = feature.split('_X_')
                # If both parts of the interaction are activated in our conditions, activate the interaction
                if conditions.get(parts[0], 0) == 1 and conditions.get(parts[1], 0) == 1:
                    vector[i] = 1
        return vector

    # Fill vectors for both groups
    vector_A = fill_vector(vector_A, group_A)
    vector_B = fill_vector(vector_B, group_B)
    
    # Create the contrast vector (Group A - Group B)
    contrast = vector_A - vector_B
    
    # Run the statistical test
    test_result = model.t_test(contrast)
    
    # Extract results cleanly
    p_value = test_result.pvalue.item()
    diff_coef = test_result.effect.item()
    odds_ratio = np.exp(diff_coef)
    
    print("==========================================================")
    print("   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)")
    print("==========================================================")
    print(f"Cell A Conditions : {group_A}")
    print(f"Cell B Conditions : {group_B}")
    print("----------------------------------------------------------")
    print(f"P-Value           : {p_value:.5f} " + ("(SIGNIFICANT ⭐️)" if p_value < 0.05 else "(NOT Significant)"))
    print(f"Relative Risk (OR): {odds_ratio:.4f} (Cell A is {odds_ratio:.2f}x riskier than Cell B)")
    print("==========================================================\n")


## import model

In [2]:
import pickle
import statsmodels.api as sm 
import pandas as pd
import numpy as np

# retrive model
with open('gee_model_interact.pkl', 'rb') as f:
    gee_model_interact = pickle.load(f)

print("Model successfully imported")

# 检查一下是否加载成功
print(gee_model_interact.summary())

Model successfully imported
                                GEE Regression Results                                
Dep. Variable:        ['deaths', 'survivors']   No. Observations:                  828
Model:                                    GEE   No. clusters:                      693
Method:                           Generalized   Min. cluster size:                   1
                         Estimating Equations   Max. cluster size:                  31
Family:                              Binomial   Mean cluster size:                 1.2
Dependence structure:            Independence   Num. iterations:                     2
Date:                        Mon, 20 Apr 2026   Scale:                           1.000
Covariance type:                       robust   Time:                         10:07:26
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const        

## --- How to use it ---
* Scenario 1: Is feeding ONLY sugar + summer treatment (Cell A) significantly different from feeding ONLY honey + no treatment (Cell B)?"*

* Note: If a feature is NOT in the dictionary, it defaults to 0. 
* If they are using the baseline features (e.g., food_2.0), just don't include them in the dict.

In [ ]:
cell_A = {'food_1.0': 1, 'varroa_4': 1}  # Sugar + summer
cell_B = {'food_3.0': 1, 'varroa_5': 1}  # Honey + no treatment

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_1.0': 1, 'varroa_4': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_5': 1}
----------------------------------------------------------
P-Value           : 0.25390 (NOT Significant)
Relative Risk (OR): 0.5671 (Cell A is 0.57x riskier than Cell B)



In [4]:
cell_A = {'food_3.0': 1, 'varroa_4': 1}  # honey + summer
cell_B = {'food_3.0': 1, 'varroa_3': 1}  # honey + winter

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_4': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_3': 1}
----------------------------------------------------------
P-Value           : 0.71002 (NOT Significant)
Relative Risk (OR): 0.7837 (Cell A is 0.78x riskier than Cell B)



In [12]:
cell_A = {'food_3.0': 1}  # honey + alter
cell_B = {'food_3.0': 1, 'varroa_3': 1}  # honey + winter

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_3': 1}
----------------------------------------------------------
P-Value           : 0.50984 (NOT Significant)
Relative Risk (OR): 0.6634 (Cell A is 0.66x riskier than Cell B)



In [13]:
cell_A = {'food_3.0': 1}  # honey + alter
cell_B = {'food_3.0': 1, 'varroa_4': 1}  # honey + summer

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_4': 1}
----------------------------------------------------------
P-Value           : 0.81233 (NOT Significant)
Relative Risk (OR): 0.8465 (Cell A is 0.85x riskier than Cell B)



In [14]:
cell_A = {'food_3.0': 1}  # honey + alter
cell_B = {'food_3.0': 1, 'varroa_5': 1}  # honey + no treatment

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_5': 1}
----------------------------------------------------------
P-Value           : 0.80625 (NOT Significant)
Relative Risk (OR): 1.1393 (Cell A is 1.14x riskier than Cell B)



In [18]:
cell_A = {'food_3.0': 1, 'varroa_1': 1}  # honey + 3 gangenmenu
cell_B = {'food_3.0': 1}  # honey + alt

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1}
----------------------------------------------------------
P-Value           : 0.00740 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1659 (Cell A is 0.17x riskier than Cell B)



In [17]:
cell_A = {'food_3.0': 1, 'varroa_1': 1}  # honey + 3 gangenmenu
for varroa_x in ['varroa_3', 'varroa_4', 'varroa_5']:
    cell_B = {'food_3.0': 1, varroa_x :1}  # honey + no treatment
    print(f"\n>>> Executing comparison for: {varroa_x} vs Baseline")
    # Run the comparison!
    compare_two_cells(gee_model_interact, cell_A, cell_B)


    
    


>>> Executing comparison for: varroa_3 vs Baseline
   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_3': 1}
----------------------------------------------------------
P-Value           : 0.00057 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1100 (Cell A is 0.11x riskier than Cell B)


>>> Executing comparison for: varroa_4 vs Baseline
   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_4': 1}
----------------------------------------------------------
P-Value           : 0.00620 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1404 (Cell A is 0.14x riskier than Cell B)


>>> Executing comparison for: varroa_5 vs Baseline
   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_5': 1}
---------------------------------------------

In [8]:
cell_A = {'food_3.0': 1, 'varroa_1': 1}  # honey + 3 gangenmenu
cell_B = {'food_3.0': 1, 'varroa_4': 1}  # honey + summer

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_4': 1}
----------------------------------------------------------
P-Value           : 0.00620 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1404 (Cell A is 0.14x riskier than Cell B)



In [9]:
cell_A = {'food_3.0': 1, 'varroa_1': 1}  # honey + 3 gangenmenu
cell_B = {'food_3.0': 1, 'varroa_5': 1}  # honey + nee

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {'food_3.0': 1, 'varroa_1': 1}
Cell B Conditions : {'food_3.0': 1, 'varroa_5': 1}
----------------------------------------------------------
P-Value           : 0.00489 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1890 (Cell A is 0.19x riskier than Cell B)



In [11]:
cell_A = {}  # mixed + alt
cell_B = {'food_3.0': 1, 'varroa_5': 1}  # honey + no treatment

# Run the comparison!
compare_two_cells(gee_model_interact, cell_A, cell_B)

   POST-HOC PAIRWISE COMPARISON (CELL A vs CELL B)
Cell A Conditions : {}
Cell B Conditions : {'food_3.0': 1, 'varroa_5': 1}
----------------------------------------------------------
P-Value           : 0.00008 (SIGNIFICANT ⭐️)
Relative Risk (OR): 0.1818 (Cell A is 0.18x riskier than Cell B)

